[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# Dates and Times


## What you will be able to do

Turn times written as text into dates and times, show the readings on Oslo's clocks through both of
2025's clock changes, and average them by day, week or rolling window with `resample` and `rolling`.

## The idea

### The problem

A file of readings stores its times as text, such as `2025-03-30T01:00`. As text, a time can be
sorted only as a string, and it has no month, no weekday and no way to ask for the hours between two
others. Some files write the day first, as `03/04/2025`, and some the month first, so the same text
means 3 April in Oslo and 4 March in New York, and nothing in the text says which.

A time also needs a time zone to name a single moment. The readings' times have none, and this
notebook takes them to be UTC, which does not change with the seasons. A person in Oslo reads a
clock that does change, which went forward an hour on 30 March 2025 and back an hour on 26 October.
On Oslo's clocks, 30 March lasted 23 hours, 26 October lasted 25, and one hour of local time
happened twice. A daily mean worked out over UTC days is not the daily mean someone in Oslo would
compute, and a local time of 02:30 on 30 March never happened at all.

### What a datetime column is

> A pandas **Timestamp** is a date and a time of day, stored to the nanosecond. A column of them has
> the type `datetime64[ns]`, or `datetime64[ns, Europe/Oslo]` when it is **time zone aware**, tied
> to a named zone, so that each value names a single moment. A column without a zone is **naive**,
> and its times name no moment until a zone is attached. **Resampling** groups the values of a
> series indexed by time into regular periods, such as days, and combines each period into one
> value.

### Why it works that way

`pd.to_datetime` turns text into Timestamps. Given a `format`, it reads every value with it. Without
one, pandas 2 works out a format from the first value and applies it to the rest, so a column that
mixes formats raises at the first value that does not fit, which is better than guessing each value
on its own.

`tz_localize` attaches a zone to naive times without moving them, and says what zone they were
recorded in. `tz_convert` takes an aware time to another zone and shows the same moment on that
zone's clocks. Converting from UTC is always possible, because every UTC moment is exactly one local
time. Localizing local times can fail, because a local time can fall in the hour the clocks skipped
or in the hour they repeated, and pandas raises rather than pick.

With a datetime column, `.dt` reaches the parts of each value: `.dt.month`, `.dt.hour`,
`.dt.day_name()` and more. With a datetime index, `resample("D")` makes a group for each day, and
`rolling` makes a window that slides along the series one value at a time.

### Where this shows up

Every log, sensor feed and transaction record has times, and most arrive as text. The **Why Pandas**
notebook used `.dt.month` to make monthly means, and the **Reshaping** notebook used `.dt.hour`.
JSON has no type for a time, so an API sends times as text, often in ISO 8601, the form the readings
use, as the weather service in the **APIs and JSON** guide does. The **sqlite3, Deep Dive** guide
stores the same readings with their hours as text, because SQLite has no type for a time.

### What this notebook covers

- times read from a file as text, and parsed with a `format`
- the parts of a time through `.dt`
- the readings in UTC and on Oslo's clocks, through both clock changes
- daily means with `resample`, in UTC and in local time
- a slice by date, and a rolling 24-hour mean
- six errors, one of them a silent misreading of dates written day first

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things.

```python
import pandas as pd

times = pd.to_datetime(pd.Series(["2025-03-30 00:00", "2025-03-30 01:00"]))
print(times.dt.tz_localize("UTC").dt.tz_convert("Europe/Oslo"))
```

```
0   2025-03-30 01:00:00+01:00
1   2025-03-30 03:00:00+02:00
dtype: datetime64[ns, Europe/Oslo]
```

The two times are an hour apart. On Oslo's clocks they read 01:00 and 03:00, because the clocks went
forward from 02:00 to 03:00 local time between them, and the offset after each time, `+01:00` and
then `+02:00`, says how far ahead of UTC Oslo was at that moment.

## Setup

Two modules for a scratch folder, one library, and three helpers for building the readings.

- `pandas`, imported as `pd`, is the library, and the cell prints its version
- `math`, `datetime` and `timedelta` build the readings
- `Path` names the scratch folder and the file in it, and `shutil` removes the folder at the end of
  Common errors

`year_of_readings` makes a reading for every hour of 2025 at the four stations in `STATIONS`, by the
same formula as the **sqlite3, Deep Dive** guide, with Svalbard's 24 readings for 2 March missing.
`readings` holds all 35,040, with `time` already parsed, and `stations` holds the latitudes. The
last lines write the readings to `scratch/readings.csv` with their times as text, which is how the
first worked example receives them.

In [1]:
import math
import shutil
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
readings.to_csv(SCRATCH / "readings.csv", index=False, date_format="%Y-%m-%dT%H:%M")   # times as text

print("pandas", pd.__version__, "|", len(readings), "readings,", len(stations), "stations, and", SCRATCH / "readings.csv")

pandas 2.2.3 | 35040 readings, 5 stations, and scratch/readings.csv


## Worked examples

### Times read from a file are text

`read_csv` does not parse times unless asked, so `time` comes back as text:

In [2]:
logged = pd.read_csv(SCRATCH / "readings.csv")

print(logged.dtypes)
try:
    logged["time"].dt.month
except AttributeError as error:
    print("AttributeError:", error)

station     object
time        object
celsius    float64
dtype: object
AttributeError: Can only use .dt accessor with datetimelike values


`object` is the type pandas gives a column of text, and `.dt`, which reaches the parts of a date,
refuses a column that holds none.

### Parsing with a format

`pd.to_datetime` with a `format` reads every value the same way. `%Y-%m-%dT%H:%M` is the year, the
month and the day, a literal `T`, then the hour and the minute:

In [3]:
parsed = pd.to_datetime(logged["time"], format="%Y-%m-%dT%H:%M")

print(parsed.dtype)
print(parsed.equals(readings["time"]))

datetime64[ns]
True


The parsed column is `datetime64[ns]`, and it equals the `time` column that Setup parsed the same
way. Text that cannot be read raises an error, unless `errors="coerce"` asks for `NaT`, pandas'
missing time, in its place:

In [4]:
pd.to_datetime(pd.Series(["2025-03-01 06:00", "not recorded"]), errors="coerce")

0   2025-03-01 06:00:00
1                   NaT
dtype: datetime64[ns]

### The parts of a time

`.dt` gives each part of every value at once:

In [5]:
time = readings["time"]

pd.DataFrame({"time": time, "month": time.dt.month, "day": time.dt.day_name(), "hour": time.dt.hour}).head(3)

,time,month,day,hour
0,2025-01-01,1,Wednesday,0
1,2025-01-01,1,Wednesday,0
2,2025-01-01,1,Wednesday,0


The first three readings are three stations at midnight on 1 January 2025, a Wednesday. The table
shows `time` without its hours, because every time in it is at midnight.

### UTC and Oslo's clocks

`tz_localize("UTC")` says the naive times were UTC, and `tz_convert("Europe/Oslo")` shows each
moment on Oslo's clocks:

In [6]:
oslo = readings[readings["station"] == "Oslo"].copy()
oslo["local"] = oslo["time"].dt.tz_localize("UTC").dt.tz_convert("Europe/Oslo")

print(oslo["local"].dtype)
oslo.head(2)

datetime64[ns, Europe/Oslo]


,station,time,celsius,local
1,Oslo,2025-01-01 00:00:00,-3.5,2025-01-01 01:00:00+01:00
5,Oslo,2025-01-01 01:00:00,-5.3,2025-01-01 02:00:00+01:00


`.copy()` makes `oslo` a table of its own, so adding a column to it gives no
`SettingWithCopyWarning`, which the **Adding and Changing Columns** notebook explains. In winter
Oslo is one hour ahead of UTC, so midnight UTC is 01:00 in Oslo, and the `+01:00` after each local
time says so.

### The clock changes in March and October

On 30 March the clocks went forward at 02:00 local time, which Oslo's readings from midnight to
02:00 UTC show:

In [7]:
oslo[oslo["time"].between("2025-03-30 00:00", "2025-03-30 02:00")]

,station,time,celsius,local
8449,Oslo,2025-03-30 00:00:00,1.7,2025-03-30 01:00:00+01:00
8453,Oslo,2025-03-30 01:00:00,1.5,2025-03-30 03:00:00+02:00
8457,Oslo,2025-03-30 02:00:00,-0.2,2025-03-30 04:00:00+02:00


The readings are an hour apart in UTC, and on Oslo's clocks 01:00 is followed by 03:00. On 26
October the clocks went back at 03:00 local time, so 02:00 came twice, first with `+02:00` and then
with `+01:00`:

In [8]:
oslo[oslo["time"].between("2025-10-25 23:00", "2025-10-26 02:00")]

,station,time,celsius,local
28605,Oslo,2025-10-25 23:00:00,4.5,2025-10-26 01:00:00+02:00
28609,Oslo,2025-10-26 00:00:00,2.5,2025-10-26 02:00:00+02:00
28613,Oslo,2025-10-26 01:00:00,2.3,2025-10-26 02:00:00+01:00
28617,Oslo,2025-10-26 02:00:00,2.3,2025-10-26 03:00:00+01:00


Counting Oslo's readings for each local day finds the days that do not hold 24 of them:

In [9]:
per_day = oslo["local"].dt.date.value_counts().sort_index()

per_day[per_day != 24]

local
2025-01-01    23
2025-03-30    23
2025-10-26    25
2026-01-01     1
Name: count, dtype: int64

30 March had 23 hours and 26 October had 25. The other two are the ends of the year. The first hour
of 1 January on Oslo's clocks is 23:00 UTC on 31 December 2024, which is not in the readings, and
the last reading, 23:00 UTC on 31 December 2025, is already 1 January 2026 in Oslo.

### Daily means with resample

`resample` needs the times as the index. Here are Oslo's readings indexed by UTC time, and the same
readings indexed by local time:

In [10]:
by_utc = oslo.set_index("time")["celsius"]
by_local = oslo.set_index("local")["celsius"]

daily_utc = by_utc.resample("D").mean()
daily_local = by_local.resample("D").mean()

print(len(daily_utc), "UTC days,", len(daily_local), "local days")
print("1 January:", round(daily_utc.iloc[0], 2), "in UTC days,", round(daily_local.iloc[0], 2), "in local days")

365 UTC days, 366 local days
1 January: -2.15 in UTC days, -2.05 in local days


`"D"` asks for a period of one day, `"h"` would ask for an hour, `"W"` a week ending on Sunday and
`"ME"` a calendar month. The local version has 366 days, because the last reading falls on 1 January
2026 in Oslo. The two means for 1 January differ, -2.15 against -2.05, because a local day starts an
hour earlier than a UTC day and holds a different set of readings. Which one is right depends on the
question, and a report for people in Oslo would use local days.

`size` counts the readings in each period, and on Oslo's clocks the two days of the clock changes
hold 23 and 25:

In [11]:
by_local.resample("D").size().loc[["2025-03-30", "2025-10-26"]]

local
2025-03-30 00:00:00+01:00    23
2025-10-26 00:00:00+02:00    25
Name: celsius, dtype: int64

For every station at once, group first and then resample, which gives a daily mean for each of the
four stations, 1,460 in all. Svalbard's silent day comes out `NaN`:

In [12]:
daily = readings.set_index("time").groupby("station")["celsius"].resample("D").mean().round(2)

print(len(daily))
daily.loc["Svalbard"].loc["2025-03-01":"2025-03-03"]

1460


time
2025-03-01   -11.11
2025-03-02      NaN
2025-03-03   -10.94
Freq: D, Name: celsius, dtype: float64

### A slice by date and a rolling mean

With a datetime index, a date written as text selects every reading on that day, and a month selects
the whole month:

In [13]:
print(len(by_utc.loc["2025-07-01"]), "readings on 1 July")
print(len(by_utc.loc["2025-07"]), "readings in July")

24 readings on 1 July
744 readings in July


`rolling(24)` takes a window of the last 24 readings at every point, so each value is the mean of
the day that ends there:

In [14]:
by_utc.rolling(24).mean().round(2).iloc[22:25]

time
2025-01-01 22:00:00     NaN
2025-01-01 23:00:00   -2.15
2025-01-02 00:00:00   -2.21
Name: celsius, dtype: float64

The first 23 windows hold fewer than 24 readings, so they are `NaN`. The window that ends at 23:00
on 1 January holds the whole of that UTC day, and its -2.15 is the first daily mean above. After
that the window slides one hour at a time, which `resample` does not do.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/16-dates-and-times-solutions.ipynb).

**1.** Parse the times `"31/12/2025 23:00"` and `"01/01/2026 00:00"`, which are written day first, with an explicit `format`, and print how far apart they are by subtracting the first from the second.

In [15]:
# your code here

**2.** Count Tromso's readings taken from 12:00 to 14:00 UTC, both hours included, using `.dt.hour`.

In [16]:
# your code here

**3.** Resample Bergen's readings to weekly means with `"W"`, and print the warmest week and its mean.

In [17]:
# your code here

**4.** Make a series of Oslo's readings indexed by local time, and print the local days that do not have 24 readings, with `resample("D").size()`.

In [18]:
# your code here

**5.** Make a 7-day rolling mean of Oslo's daily means in UTC days, and print its lowest value and the day its window ends on.

In [19]:
# your code here

## Common errors

### ValueError: time data "03/01/2025" doesn't match format "%Y-%m-%d", at position 1.

In [20]:
pd.to_datetime(pd.Series(["2025-01-03", "03/01/2025"]))

ValueError: time data "03/01/2025" doesn't match format "%Y-%m-%d", at position 1. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

Without a `format`, pandas 2 worked out `%Y-%m-%d` from the first value, and the second value,
written another way, did not fit. The rest of the message lists the ways out. `format="mixed"` reads
each value on its own, and with `dayfirst=True` it reads `03/01/2025` as 3 January:

In [21]:
pd.to_datetime(pd.Series(["2025-01-03", "03/01/2025"]), format="mixed", dayfirst=True)

0   2025-01-03
1   2025-01-03
dtype: datetime64[ns]

### No error, and 3 April read as 4 March

In [22]:
pd.to_datetime(pd.Series(["03/04/2025", "04/05/2025"]))

0   2025-03-04
1   2025-04-05
dtype: datetime64[ns]

A Norwegian logger writes the day first, so these are 3 April and 4 May. pandas read the month
first, and both values fit that reading, so nothing failed and both dates are wrong. Name the order,
with `format="%d/%m/%Y"` or `dayfirst=True`, whenever the day can come first:

In [23]:
pd.to_datetime(pd.Series(["03/04/2025", "04/05/2025"]), format="%d/%m/%Y")

0   2025-04-03
1   2025-05-04
dtype: datetime64[ns]

### NonExistentTimeError: 2025-03-30 02:30:00

In [24]:
pd.Series(pd.to_datetime(["2025-03-30 02:30"])).dt.tz_localize("Europe/Oslo")

NonExistentTimeError: 2025-03-30 02:30:00

This time was written down as Oslo local time, and on 30 March Oslo's clocks went from 01:59 to
03:00, so 02:30 never happened there. The full name of the error is
`pytz.exceptions.NonExistentTimeError`, from pytz, the time zone library pandas uses for a zone
named by text. `nonexistent="shift_forward"` moves such a time to the first moment that exists:

In [25]:
pd.Series(pd.to_datetime(["2025-03-30 02:30"])).dt.tz_localize("Europe/Oslo", nonexistent="shift_forward")

0   2025-03-30 03:00:00+02:00
dtype: datetime64[ns, Europe/Oslo]

### AmbiguousTimeError: Cannot infer dst time from 2025-10-26 02:30:00, try using the 'ambiguous' argument

In [26]:
pd.Series(pd.to_datetime(["2025-10-26 02:30"])).dt.tz_localize("Europe/Oslo")

AmbiguousTimeError: Cannot infer dst time from 2025-10-26 02:30:00, try using the 'ambiguous' argument

On 26 October, 02:30 happened twice in Oslo, once in summer time and once in winter time, and the
text does not say which. Times recorded in UTC have neither problem, which is the reason to log in
UTC and convert for display. For local times already written down, `ambiguous=[True]` says each was
the first, in summer time, with one `True` or `False` for each value:

In [27]:
pd.Series(pd.to_datetime(["2025-10-26 02:30"])).dt.tz_localize("Europe/Oslo", ambiguous=[True])

0   2025-10-26 02:30:00+02:00
dtype: datetime64[ns, Europe/Oslo]

### TypeError: Invalid comparison between dtype=datetime64[ns, UTC] and Timestamp

In [28]:
in_utc = readings["time"].dt.tz_localize("UTC")

in_utc > pd.Timestamp("2025-07-01")

TypeError: Invalid comparison between dtype=datetime64[ns, UTC] and Timestamp

`in_utc` is aware and `pd.Timestamp("2025-07-01")` is naive, and pandas will not compare a moment
with a time that names no zone. Give the comparison the same zone:

In [29]:
print((in_utc > pd.Timestamp("2025-07-01", tz="UTC")).sum(), "readings after midnight UTC on 1 July")

17660 readings after midnight UTC on 1 July


### TypeError: Only valid with DatetimeIndex, TimedeltaIndex or PeriodIndex, but got an instance of 'Index'

In [30]:
oslo.resample("D").mean()

TypeError: Only valid with DatetimeIndex, TimedeltaIndex or PeriodIndex, but got an instance of 'Index'

`resample` groups by the index, and `oslo`'s index holds the row labels it kept from `readings`, 1,
5, 9 and so on, which are not times. Set the time column as the index first, as the worked examples
did, or name the column with `on`:

In [31]:
oslo.resample("D", on="time")["celsius"].mean().round(2).head(2)

time
2025-01-01   -2.15
2025-01-02   -2.22
Freq: D, Name: celsius, dtype: float64

The notebook is finished with its file, so this cell removes the scratch folder:

In [32]:
shutil.rmtree(SCRATCH)

print("scratch still there:", SCRATCH.exists())

scratch still there: False


## Recap

- Parse times from text with `pd.to_datetime` and an explicit `format`, and name the day's position
  when it comes first
- Read a time's parts with `.dt`, such as `.dt.month` and `.dt.hour`
- Attach a zone with `tz_localize` and show another zone's clocks with `tz_convert`
- Average by day or week with `resample`, and over a sliding window with `rolling`
- Recognize the hours the clocks skip and repeat, and keep times in UTC until they are shown

## What is next

The **Strings** notebook works on text columns with the `.str` accessor, cleaning station names
typed several ways and pulling each day's low and high out of the technicians' notes.

---

&#8592; **Previous:** [Merging](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/15-merging.ipynb)  &nbsp;·&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Strings](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/17-strings.ipynb) &#8594;
